<a href="https://colab.research.google.com/github/advayc/tencent-auk/blob/main/tencent_auk_audio_editor_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Tencent AuK Audio Editor — Google Colab

Edit audio that you own or are authorized to edit with **Tencent AuK**. This notebook covers:

1. **Transcript editing** — replace, insert, or remove spoken words while preserving the voice.
2. **Tone / emotion editing** — make delivery calm, excited, happy, etc.
3. **Speech cleanup** — reduce noise and room reverb, then restore clearer speech.
4. **Speaker isolation** — keep one speaker from a multi-speaker recording.

AuK generates a new edited waveform, so always listen to the result before publishing it. Do not use this notebook to misrepresent a person or create a deceptive recording.

## Before you run it

In Colab choose **Runtime → Change runtime type → GPU**. Use an **Ampere-or-newer bfloat16 GPU**, preferably an **L4 (24 GB) or A100 (40 GB+)**. A T4 is not suitable: AuK uses bfloat16 and current upstream validation reports FP16 instability.

The first run downloads roughly 20 GB of model assets and needs at least 25 GB of free runtime storage. The notebook uses current upstream AuK with CPU offload enabled to reduce GPU usage. It is best to test short clips first (about 5–20 seconds). The default **Base** model prioritizes quality; choose **Flash** below if you prefer speed.

In [ ]:
# Run this first. It avoids importing torch into the notebook kernel before installation.
import json
import shutil
import subprocess
import sys

if sys.version_info < (3, 10):
    raise RuntimeError(f'AuK needs Python 3.10+; this runtime is {sys.version.split()[0]}.')
probe_code = (
    'import json, torch; '
    'print(json.dumps({\"cuda\": torch.cuda.is_available(), '
    '\"bf16\": torch.cuda.is_bf16_supported() if torch.cuda.is_available() else False, '
    '\"name\": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None, '
    '\"vram\": torch.cuda.get_device_properties(0).total_memory / 2**30 if torch.cuda.is_available() else 0}))'
)
probe = json.loads(subprocess.check_output([sys.executable, '-c', probe_code], text=True))
if not probe['cuda']:
    raise RuntimeError('No CUDA GPU found. In Colab select Runtime → Change runtime type → GPU, then reconnect.')
if not probe['bf16']:
    raise RuntimeError(
        'This GPU does not support bfloat16. Use an Ampere-or-newer GPU such as an L4 or A100; a T4 will not run AuK reliably.'
    )

gpu_vram_gib = probe['vram']
free_disk_gib = shutil.disk_usage('/content').free / 2**30
print(f"GPU: {probe['name']} ({gpu_vram_gib:.1f} GiB VRAM)")
print(f'Free /content storage: {free_disk_gib:.1f} GiB')

if gpu_vram_gib < 20:
    raise RuntimeError('Use a GPU with at least 20 GiB VRAM. An L4 (24 GiB) or A100 is recommended.')
if free_disk_gib < 25:
    raise RuntimeError('Free at least 25 GiB in /content, then restart the runtime and try again.')


## Choose a model and install AuK

Base is selected by default because Tencent documents it as the higher-quality variant. Flash is much faster and uses a fixed four-step recipe. The notebook downloads and loads only one variant at a time. Run the install cell only once per fresh Colab runtime.

In [ ]:
#@title Choose model (select before downloading weights)
model_variant = "base" #@param ["base", "flash"]

MODEL_REPO = 'tencent/AuK-Flash' if model_variant == 'flash' else 'tencent/AuK'
MODEL_DIR = '/content/ckpts/AuK-Flash' if model_variant == 'flash' else '/content/ckpts/AuK'
MODEL_FILENAME = 'auk_flash.safetensors' if model_variant == 'flash' else 'auk_base.safetensors'
print(f'Selected {model_variant}: {MODEL_REPO}')


In [ ]:
%cd /content
!if [ ! -d /content/AuK/.git ]; then git clone --depth 1 --branch main https://github.com/Tencent-Hunyuan/AuK.git /content/AuK; else git -C /content/AuK pull --ff-only; fi
%cd /content/AuK
!pip -q install -U "huggingface_hub[cli]"
!pip -q install -e .

# Colab already includes ffmpeg. This confirms it is available for input normalization.
!ffmpeg -version | head -n 1
!python -c "import torch, torchaudio, transformers; from auk.infer.infer_auk import AukInfer, save_audio; print(f'AuK import succeeded — torch {torch.__version__}, torchaudio {torchaudio.__version__}, transformers {transformers.__version__}')"


## Download model weights

Downloads are stored only in this Colab runtime and disappear after a runtime reset. If Hugging Face returns a 401 or 403, run `from huggingface_hub import login; login()` in a separate cell and provide your own read-only Hugging Face token — never paste a token into this notebook.

In [ ]:
%cd /content
!mkdir -p /content/ckpts
!hf download {MODEL_REPO} --local-dir {MODEL_DIR}
!hf download Qwen/Qwen2.5-Omni-3B --local-dir /content/ckpts/Qwen2.5-Omni-3B

!test -f {MODEL_DIR}/{MODEL_FILENAME}
!test -f {MODEL_DIR}/config.yaml
!du -sh {MODEL_DIR} /content/ckpts/Qwen2.5-Omni-3B


## Load the selected AuK model

Loading can take a few minutes. `cpu_offload=True` trades some speed for much lower peak VRAM use. Keep this cell's `dtype='bf16'`; changing it to FP16 is not a safe workaround for a T4.

In [ ]:
from pathlib import Path

import torch

from auk.infer.infer_auk import AukInfer, get_gen_duration, save_audio

REPO_DIR = Path('/content/AuK')
CHECKPOINT_DIR = Path(MODEL_DIR)
QWEN_DIR = Path('/content/ckpts/Qwen2.5-Omni-3B')
INPUT_DIR = Path('/content/auk_input')
OUTPUT_DIR = Path('/content/auk_outputs')
INPUT_DIR.mkdir(exist_ok=True)
OUTPUT_DIR.mkdir(exist_ok=True)

engine = AukInfer(
    str(CHECKPOINT_DIR / 'config.yaml'),
    str(CHECKPOINT_DIR / MODEL_FILENAME),
    qwen_path=str(QWEN_DIR),
    device='cuda:0',
    dtype='bf16',
    cpu_offload=True,
)
print(f'AuK {model_variant} is ready.')


## Upload one source recording

Upload WAV, MP3, M4A, or another format that ffmpeg can read. The notebook converts it to a mono 24 kHz WAV before inference; your original upload stays intact. For transcript edits, write down the exact source phrase before editing it.

In [ ]:
import subprocess

import torchaudio
from google.colab import files
from IPython.display import Audio, display

uploaded = files.upload()
if len(uploaded) != 1:
    raise ValueError('Upload exactly one source audio file, then run this cell again.')

source_name, source_bytes = next(iter(uploaded.items()))
source_raw_path = INPUT_DIR / Path(source_name).name
source_raw_path.write_bytes(source_bytes)
source_path = INPUT_DIR / 'source_24k_mono.wav'
subprocess.run(
    ['ffmpeg', '-hide_banner', '-loglevel', 'error', '-y', '-i', str(source_raw_path), '-ac', '1', '-ar', '24000', str(source_path)],
    check=True,
)

info = torchaudio.info(str(source_path))
source_seconds = info.num_frames / info.sample_rate
print(f'Source: {source_name}')
print(f'Normalized: {source_path.name} | {source_seconds:.2f} s | {info.sample_rate} Hz | {info.num_channels} channel')
if source_seconds > 20:
    print('Tip: start with a shorter 5–20 second clip for more reliable edits and lower memory use.')
display(Audio(filename=str(source_path)))


In [ ]:
def run_edit(instruction, output_name, *, gen_seconds=None):
    """Generate one AuK edit, preview it, and return its safe output path."""
    if 'source_path' not in globals() or not Path(source_path).exists():
        raise RuntimeError('Run the upload cell first.')

    output_path = OUTPUT_DIR / Path(output_name).name
    messages = [{
        'role': 'user',
        'content': [
            {'type': 'text', 'text': instruction},
            {'type': 'audio', 'audio': str(source_path)},
        ],
    }]

    with torch.inference_mode():
        audio, sample_rate = engine.generate(messages, gen_seconds=gen_seconds)
    save_audio(audio, sample_rate, str(output_path))

    print(f'Instruction: {instruction}')
    print(f'Saved: {output_path}')
    display(Audio(filename=str(output_path)))
    return output_path


## 1. Transcript editing

Use the source phrase exactly as spoken. By default AuK regenerates the original duration. For a better estimate when the replacement changes the amount of speech, provide the **complete** original and edited transcripts below—not just the phrase being replaced. You can also enter a manual target duration. For insertion/removal, replace the `instruction` line with one of the examples below.

- `Add 'the new words' after 'the anchor phrase'.`
- `Remove 'the words to remove'.`

In [ ]:
#@title Configure the transcript edit
original_words = "old words exactly as spoken" #@param {type:"string"}
replacement_words = "new words to say" #@param {type:"string"}
duration_mode = "original_duration" #@param ["original_duration", "estimate_from_full_transcripts", "manual"]
full_source_transcript = "" #@param {type:"string"}
full_edited_transcript = "" #@param {type:"string"}
manual_target_seconds = 0.0 #@param {type:"number"}

if original_words == 'old words exactly as spoken' or replacement_words == 'new words to say':
    raise ValueError('Replace the placeholder phrases with your actual source and replacement words.')

instruction = f"Replace '{original_words}' with '{replacement_words}'."
if duration_mode == 'original_duration':
    transcript_seconds = source_seconds
elif duration_mode == 'estimate_from_full_transcripts':
    if not full_source_transcript.strip() or not full_edited_transcript.strip():
        raise ValueError('Provide both complete transcripts to estimate an edited duration.')
    transcript_seconds = get_gen_duration(
        audio=str(source_path),
        ref_text=full_source_transcript,
        gen_text=full_edited_transcript,
    )
elif duration_mode == 'manual':
    if manual_target_seconds <= 0:
        raise ValueError('Enter a positive manual target duration in seconds.')
    transcript_seconds = float(manual_target_seconds)
else:
    raise ValueError(f'Unexpected duration mode: {duration_mode}')

print(f'Generating {transcript_seconds:.2f} seconds of edited audio.')
transcript_output = run_edit(
    instruction,
    '01_transcript_edit.wav',
    gen_seconds=transcript_seconds,
)


## 2. Tone and emotion editing

AuK's documented emotions are `happy`, `angry`, `sad`, `fearful`, `surprised`, `disgusted`, `calm`, and `excited`. This operation preserves the source duration.

In [ ]:
#@title Choose an emotion
target_emotion = "excited" #@param ["happy", "angry", "sad", "fearful", "surprised", "disgusted", "calm", "excited"]

emotion_output = run_edit(
    f'Change the emotion to {target_emotion}.',
    f'02_emotion_{target_emotion}.wav',
)


## 3. Speech cleanup

Choose whether to remove background noise, room reverb, or both. AuK's enhancement prompts are designed to preserve all speakers and the original duration.

In [ ]:
#@title Choose cleanup mode
cleanup_mode = "noise_and_reverb" #@param ["noise_only", "reverb_only", "noise_and_reverb"]

cleanup_instructions = {
    'noise_only': 'Remove only the background noise, preserve everything else, and output audio of the same length.',
    'reverb_only': 'Remove only the room reverberation, preserve everything else, and output audio of the same length.',
    'noise_and_reverb': 'Preserve all speakers, remove noise and reverberation, and output clean speech of the same length.',
}
cleanup_output = run_edit(
    cleanup_instructions[cleanup_mode],
    f'03_cleanup_{cleanup_mode}.wav',
)


## 4. Speaker isolation

Choose a speaker by **speaking order** (the first, second, etc. person to begin talking) or use a distinctive **phrase** they say. Overlap and very similar voices remain difficult cases, so listen to the result carefully.

In [ ]:
#@title Configure speaker isolation
isolation_method = "speaking_order" #@param ["speaking_order", "phrase"]
speaker_order = "second" #@param ["first", "second", "third", "fourth"]
speaker_phrase = "a distinctive phrase the target speaker says" #@param {type:"string"}

if isolation_method == 'speaking_order':
    speaker_instruction = (
        f'Keep only the {speaker_order} speaker to start talking and remove all other speakers.'
    )
else:
    if speaker_phrase == 'a distinctive phrase the target speaker says' or not speaker_phrase.strip():
        raise ValueError('Enter a distinctive phrase spoken by the target speaker.')
    speaker_instruction = (
        f'Keep only the speaker who says "{speaker_phrase}" and remove all other speakers.'
    )

speaker_output = run_edit(
    speaker_instruction,
    '04_speaker_isolation.wav',
)


## Download your results

Run this after one or more edit cells. It creates and downloads a ZIP containing only the outputs that were generated in this runtime.

In [ ]:
import zipfile

result_paths = [
    value for name, value in globals().items()
    if name.endswith('_output') and isinstance(value, Path) and value.exists()
]
if not result_paths:
    raise RuntimeError('No output files found. Run at least one edit cell first.')

archive_path = OUTPUT_DIR / 'auk_results.zip'
with zipfile.ZipFile(archive_path, 'w', compression=zipfile.ZIP_DEFLATED) as archive:
    for result_path in result_paths:
        archive.write(result_path, arcname=result_path.name)

print('Created:', archive_path)
files.download(str(archive_path))


## Troubleshooting and references

- **Out of memory:** restart the runtime, confirm you received an L4/A100-class bfloat16 GPU, keep `cpu_offload=True`, and try a shorter clip.
- **Model download fails:** check available disk space; if Hugging Face asks for authentication, use `huggingface_hub.login()` with your own read-only token.
- **A replacement sounds rushed or cut off:** choose `manual` duration mode and increase `manual_target_seconds`, or use the full-transcript estimate. Content editing requires an explicit target duration.
- **Output is not exact:** this is generative audio editing. Review wording, speaker identity, timing, and artifacts before use.

Official resources: [AuK repository](https://github.com/Tencent-Hunyuan/AuK) · [AuK inference cookbook](https://github.com/Tencent-Hunyuan/AuK/blob/main/docs/COOKBOOK.md) · [AuK Base](https://huggingface.co/tencent/AuK) · [AuK-Flash](https://huggingface.co/tencent/AuK-Flash). AuK is released under MIT; the required Qwen encoder has separate Qwen Research License terms, including commercial-use restrictions, so review both before use.